In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import kruskal

In [6]:
import os
path = "iHMP_IBDMDB_2019/iHMP_IBDMDB_2019/mtb.tsv"
print("Exists:", os.path.exists(path))
print("Size:", os.path.getsize(path) if os.path.exists(path) else "N/A")
print("Readable:", os.access(path, os.R_OK))

Exists: True
Size: 0
Readable: True


In [8]:
import zipfile
import os

zip_path = "iHMP_IBDMDB_2019/iHMP_IBDMDB_2019/mtb.tsv.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    print("Files inside zip:", z.namelist())
    z.extract(z.namelist()[0], path="iHMP_IBDMDB_2019/iHMP_IBDMDB_2019/extracted_temp/")

print("Done — check the extracted_temp subfolder")

Files inside zip: ['mtb.tsv']
Done — check the extracted_temp subfolder


In [9]:
metadata_hmp2 = pd.read_csv("iHMP_IBDMDB_2019/iHMP_IBDMDB_2019/metadata.tsv", sep="\t")
mtb_hmp2 = pd.read_csv("iHMP_IBDMDB_2019/iHMP_IBDMDB_2019/extracted_temp/mtb.tsv", sep="\t")
df2 = metadata_hmp2.merge(mtb_hmp2, on="Sample")

print("Joined shape:", df2.shape)
print(df2["Study.Group"].value_counts())

Joined shape: (382, 81891)
Study.Group
CD        177
nonIBD    104
UC        101
Name: count, dtype: int64


In [10]:
feature_cols2 = [c for c in mtb_hmp2.columns if c != "Sample"]
X2 = df2[feature_cols2]
y2 = df2["Study.Group"]

iso2 = IsolationForest(contamination=0.05, random_state=42)
flags2 = iso2.fit_predict(X2)

print("Outliers removed:", (flags2 == -1).sum())
print("Samples kept:", (flags2 == 1).sum())

Outliers removed: 20
Samples kept: 362


In [11]:
X2_f = X2[flags2 == 1]
y2_f = y2[flags2 == 1]

X2_log = np.log2(X2_f + 1)

print("Shape after log transform:", X2_log.shape)
print(X2_log.iloc[:3, :5])

Shape after log transform: (362, 81867)
   C18n_QI06__12.13-diHOME  C18n_QI07__9.10-diHOME  C18n_QI08__caproate  \
1                23.281225               23.148773                  NaN   
3                20.044860               19.382084            10.553629   
4                18.276029               19.071279            15.424265   

   C18n_QI09__heptanoate  C18n_QI10__hydrocinnamate  
1                    NaN                  12.334553  
3                    NaN                  17.569655  
4              15.231634                  20.073340  


In [12]:
missing_pct = X2_log.isna().mean().sort_values(ascending=False)
print("Features with >50% missing:", (missing_pct > 0.5).sum())
print("Features with >20% missing:", (missing_pct > 0.2).sum())
print("Features with 0% missing:", (missing_pct == 0).sum())
print("\nOverall missingness:", X2_log.isna().mean().mean())

Features with >50% missing: 31225
Features with >20% missing: 47107
Features with 0% missing: 6166

Overall missingness: 0.3858890315764142


In [13]:
# Keep only features with less than 20% missing (consistent with the same
# prevalence-filtering logic used in your other pipelines)
keep_cols = missing_pct[missing_pct <= 0.20].index
X2_log_filtered = X2_log[keep_cols]

print("Features kept:", X2_log_filtered.shape[1], "of", X2_log.shape[1])

# For the small remaining gaps (<=20% missing), fill with 0 —
# same convention as Franzosa, where undetected = zero abundance
X2_log_filtered = X2_log_filtered.fillna(0)

print("Any NaNs left?", X2_log_filtered.isna().sum().sum())
print("Final shape:", X2_log_filtered.shape)

Features kept: 34760 of 81867
Any NaNs left? 0
Final shape: (362, 34760)


In [15]:
def kruskal_top_k(X, y, k):
    groups = y.unique()
    pvals = {}
    for col in X.columns:
        samples_by_group = [X.loc[y.values == g, col] for g in groups]
        try:
            stat, p = kruskal(*samples_by_group)
        except ValueError:
            p = 1.0
        pvals[col] = p
    return sorted(pvals, key=pvals.get)[:k]

feature_sizes = [5, 10, 20, 40]
selected2 = {k: kruskal_top_k(X2_log_filtered, y2_f, k) for k in feature_sizes}

for k, feats in selected2.items():
    print(f"\nTop {k} features:")
    print(feats)


Top 5 features:
['C18n_QI10930__NA', 'C18n_QI7515__NA', 'C18n_QI382__NA', 'C18n_QI846__NA', 'C18n_QI10987__NA']

Top 10 features:
['C18n_QI10930__NA', 'C18n_QI7515__NA', 'C18n_QI382__NA', 'C18n_QI846__NA', 'C18n_QI10987__NA', 'C18n_QI11350__NA', 'C18n_QI663__NA', 'C18n_QI7434__NA', 'C18n_QI745__NA', 'C18n_QI334__NA']

Top 20 features:
['C18n_QI10930__NA', 'C18n_QI7515__NA', 'C18n_QI382__NA', 'C18n_QI846__NA', 'C18n_QI10987__NA', 'C18n_QI11350__NA', 'C18n_QI663__NA', 'C18n_QI7434__NA', 'C18n_QI745__NA', 'C18n_QI334__NA', 'C18n_QI11343__NA', 'C18n_QI740__NA', 'C18n_QI11349__NA', 'C18n_QI7605__NA', 'C18n_QI742__NA', 'C18n_QI741__NA', 'C18n_QI11341__NA', 'C18n_QI744__NA', 'C18n_QI7436__NA', 'C18n_QI1328__NA']

Top 40 features:
['C18n_QI10930__NA', 'C18n_QI7515__NA', 'C18n_QI382__NA', 'C18n_QI846__NA', 'C18n_QI10987__NA', 'C18n_QI11350__NA', 'C18n_QI663__NA', 'C18n_QI7434__NA', 'C18n_QI745__NA', 'C18n_QI334__NA', 'C18n_QI11343__NA', 'C18n_QI740__NA', 'C18n_QI11349__NA', 'C18n_QI7605__NA', 

In [16]:
le2 = LabelEncoder()
y2_enc = le2.fit_transform(y2_f)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
rf_grid = {"n_estimators": [200, 500], "max_features": ["sqrt", "log2"], "min_samples_leaf": [1, 3, 5]}
en_grid = {"C": [0.01, 0.1, 1, 10], "l1_ratio": [0.1, 0.5, 0.9]}

results2 = []
for k in feature_sizes:
    print(f"\n--- Running k={k} ---")
    Xk = X2_log_filtered[selected2[k]]

    rf = GridSearchCV(RandomForestClassifier(random_state=42), rf_grid, cv=inner_cv, scoring="roc_auc_ovr", n_jobs=-1)
    rf_probs = cross_val_predict(rf, Xk, y2_enc, cv=outer_cv, method="predict_proba")
    rf_auc = roc_auc_score(y2_enc, rf_probs, multi_class="ovr", average="macro")

    Xk_scaled = StandardScaler().fit_transform(Xk)
    en = GridSearchCV(LogisticRegression(penalty="elasticnet", solver="saga", max_iter=5000), en_grid, cv=inner_cv, scoring="roc_auc_ovr", n_jobs=-1)
    en_probs = cross_val_predict(en, Xk_scaled, y2_enc, cv=outer_cv, method="predict_proba")
    en_auc = roc_auc_score(y2_enc, en_probs, multi_class="ovr", average="macro")

    print(f"k={k}: RF macro-AUC={rf_auc:.3f} | ElasticNet macro-AUC={en_auc:.3f}")
    results2.append({"n_features": k, "model": "RandomForest", "macro_auc": rf_auc})
    results2.append({"n_features": k, "model": "ElasticNet", "macro_auc": en_auc})

results2_df = pd.DataFrame(results2)
results2_df.to_csv("hmp2_ibdpred_style_results.csv", index=False)
print("\n", results2_df)


--- Running k=5 ---
k=5: RF macro-AUC=0.796 | ElasticNet macro-AUC=0.771

--- Running k=10 ---
k=10: RF macro-AUC=0.835 | ElasticNet macro-AUC=0.803

--- Running k=20 ---
k=20: RF macro-AUC=0.847 | ElasticNet macro-AUC=0.814

--- Running k=40 ---
k=40: RF macro-AUC=0.858 | ElasticNet macro-AUC=0.826

    n_features         model  macro_auc
0           5  RandomForest   0.795565
1           5    ElasticNet   0.770892
2          10  RandomForest   0.835019
3          10    ElasticNet   0.802922
4          20  RandomForest   0.846669
5          20    ElasticNet   0.814033
6          40  RandomForest   0.857994
7          40    ElasticNet   0.825586
